# Kvantu skaitīšana (Quantum counting)

Kvantu skaitīšana ir algoritms, kas ļauj novērtēt, cik daudz “atzīmēto” elementu ir meklēšanas telpā.

Ja ir:
- $N = 2^n$ iespējamie stāvokļi (n kubiti),
- $M$ no tiem ir atzīmēti,

tad kvantu skaitīšana dod novērtējumu $\tilde M$ ar kvadrātisku ātruma ieguvumu salīdzinājumā ar klasisku paraugu ņemšanu.

Tātad:

- Amplitūdu pastiprināšanu izmanto ja $M$ ir zināms vai aptuveni zināms, izvēlamies iterāciju skaitu $k$ un pastiprinām varbūtību atrast marķētos stāvokļus.
- Kvantu skaitīšana: ja $M$ nav zināms, vispirms ar QPE uz $Q$ iegūstam $\tilde M$, un tad varam optimāli izvēlēties $k$ lai varētu realizēt Amplitūdu pastiprināšanu / Grover meklēšanu.


## Kā tas saistās ar Amplitūdu pastiprināsanu

Grovera meklēšana un Amplitūdu pastiprināšana darbojas 2D apakštelpā kā rotācija.

Sākotnējā vienmērīgā superpozīcija ir:
$$
|s\rangle = \frac{1}{\sqrt{N}}\sum_{x=0}^{N-1} |x\rangle.
$$

Oracle atzīmē marķētos stāvokļus (fāzes flips):
$$
S_f|x\rangle = (-1)^{f(x)}|x\rangle.
$$

Grover operatora viena iterācija ir:
$$
Q = -A S_0 A^\dagger S_f,
$$
kur bieži $A = H^{\otimes n}$, un $|s\rangle = A|0\rangle$.

Ja $M$ ir atzīmēto skaits, tad:
$$
\sin^2(\theta) = \frac{M}{N}.
$$

Galvenā ideja ir tāda, ka operators $Q$ darbojas kā rotācija par leņķi $2\theta$ attiecīgajā marķēto/nemarķēto plaknē.

Tā kā jebkuras rotācijas divdimensionālā plaknē īpašvērtības ir kompleksie eksponenti, tad arī šeit tās ir
$$
e^{\pm i 2\theta}.
$$

No tā seko, ka, ja mēs spējam noteikt šo fāzi $2\theta$, tad varam aprēķināt $\theta$ un līdz ar to atgūt meklēto lielumu $M$.


Kvantu skaitīšana būtībā ir Fāzes novērtēšana (QPE) uz Grovera operatoru $Q$.

Izmantojot QPE, iespējams novērtēt fāzi $\varphi$, kur:
$$
e^{2\pi i \varphi} = e^{i2\theta}
\quad\Rightarrow\quad
\varphi = \frac{\theta}{\pi}
\quad (\text{mod } 1).
$$

No QPE rezultāta iegūstam $\tilde\theta$, un tad aprēķinām:
$$
\tilde M = N\sin^2(\tilde\theta).
$$


In [4]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.library import QFT, GroverOperator
import numpy as np

#fāzes apgriezšana konkrētam 3-bit stāvoklim
def phase_flip(qc, qubits, bitstring):
    q0, q1, q2 = qubits
    bits = [int(b) for b in bitstring] #pārvērš tekstu par masīvu ar bita vērtībām

    # Pārveido mērķa stāvokli uz |111> (tur, kur bit = 0, uzliek X)
    if bits[0] == 0: qc.x(q0)
    if bits[1] == 0: qc.x(q1)
    if bits[2] == 0: qc.x(q2)

    qc.h(q2)
    qc.ccx(q0, q1, q2)
    qc.h(q2)

    if bits[0] == 0: qc.x(q0)
    if bits[1] == 0: qc.x(q1)
    if bits[2] == 0: qc.x(q2)


# Orākuls atzīmē divus stāvokļus ar fāzi -1
oracle = QuantumCircuit(3, name="Oracle")
phase_flip(oracle, [0, 1, 2], "101")
phase_flip(oracle, [0, 1, 2], "110")

#Uztaisa Grover operatoru Q no orākula
Q = GroverOperator(oracle).to_gate(label="Q")

t = 6         # counting kubiti (precizitātei)
n = 3          # search kubiti
N = 2 ** n

qc = QuantumCircuit(t + n, t)

counting = list(range(t))         
search = list(range(t, t + n))  

# Abi reģistri superpozīcijā
qc.h(counting)
qc.h(search)

# Controlled-Q^(2^k)
cQ = Q.control(1) #pārveido Grovera operātoru par kontrolētu
for k in range(t):
    reps = 2 ** k
    for rep in range(reps):
        qc.append(cQ, [counting[k]] + search) # counting[k] ir kontroles kubits, search ir mērķa kubiti

# Inverse QFT uz counting reģistra
qc.append(QFT(t).inverse().to_gate(label="QFT†"), counting)

qc.measure(counting, list(range(t)))

print(qc.draw(fold=120))


sim = AerSimulator()
tqc = transpile(qc, sim)
result = sim.run(tqc, shots=4000).result()
counts = result.get_counts()

print("Counting mērījumi:", counts)

# Paņem biežāko bitu virkni kā fāzes novērtējumu
top = max(counts, key=counts.get)      
y = int(top, 2)
phi = y / (2 ** t)                      # phi ~ theta/pi
M_est = N * (np.sin(np.pi * phi) ** 2)  # M ≈ N sin^2(pi*phi)

print(f"Top = {top}, phi ≈ {phi:.4f}, M_est ≈ {M_est:.2f} (īstais M=2, N=8)")

C:\Users\ovase\AppData\Local\Temp\ipykernel_8180\1851748471.py:31: DeprecationWarning: The class ``qiskit.circuit.library.grover_operator.GroverOperator`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use qiskit.circuit.library.grover_operator instead.
  Q = GroverOperator(oracle).to_gate(label="Q")
C:\Users\ovase\AppData\Local\Temp\ipykernel_8180\1851748471.py:54: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  qc.append(QFT(t).inverse().to_gate(label="QFT†"), counting)


     ┌───┐                                                                                                            »
q_0: ┤ H ├──■─────────────────────────────────────────────────────────────────────────────────────────────────────────»
     ├───┤  │                                                                                                         »
q_1: ┤ H ├──┼─────■─────■─────────────────────────────────────────────────────────────────────────────────────────────»
     ├───┤  │     │     │                                                                                             »
q_2: ┤ H ├──┼─────┼─────┼─────■─────■─────■─────■─────────────────────────────────────────────────────────────────────»
     ├───┤  │     │     │     │     │     │     │                                                                     »
q_3: ┤ H ├──┼─────┼─────┼─────┼─────┼─────┼─────┼─────■─────■─────■─────■─────■─────■─────■─────■─────────────────────»
     ├───┤  │     │     │     │     │   